# **ULTIMATE STOCK SCREENER | UNIFIED MARKET INTELLIGENCE DASHBOARD**

Dashboard features:
* Automatic Discovery
* Unusual Volume Alerts
* Advanced Fundamental Filters
* Clickable Ticker Buttons
* Interactive Technical Charts

Run cells in order. Data comes from Yahoo Finance via yfinance; quotes and financial statements may be delayed,
incomplet, or contain rate-limits. The default sample is 1500 randomly selected eligible US listings, not the entire 
market. Rankings and alerts apply only to successfully scanned symbols. 
**This notebook is for informational use only. Not trading recommendations**

In [14]:
%pip -q install -U yfinance ipywidgets requests lxml html5lib

Note: you may need to restart the kernel to use updated packages.


In [15]:
import io, time, warnings, requests
import numpy as np, pandas as pd, yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 30)
print('Ready. Run the discovery cell, then the dashboard.')

Ready. Run the discovery cell, then the dashboard.


# **Automatically Discover A Broad US Stock Universe**

This uses the Nasdaq Trader symbol directories (Nasdaq-listed and other US-listed stocks). Excludes ETFs, test issues and obvious special securities.
To avoid thousands of requests and rate limits, the default scan uses a **rotating subset** of up to 500 symbols, plus Yahoo's market-mover feeds where available.
Increase 'SCAN_LIMIT' to broaden coverage, but expect slower scans. 
The tables are **top within the scanned set**, not market-wide official leaders.

**Larger Scans:** Default 1,500 symbols, downloaded in 40-ticker batches with pauses, limited retries, and a 12-hour local batch cache. Set 'SCAN_LIMIT' to 500 for faster runs or 
3000 for broader coverage. Cache is stored in the notebook working directory and may be ephemeral in Colob. Market data can be delayed or incomplete.

In [16]:
# Recommended: 1,500. Choose 500 for quick scans, 3,000 for longer scans.
# None attempts the entire directory; it may take considerable time and can hit provider limits.
SCAN_LIMIT = 1500
SCAN_BATCH_SIZE = 40
SCAN_PAUSE_SECONDS = 1.0
SCAN_MAX_RETRIES = 2
SCAN_USE_CACHE = True
SCAN_CACHE_HOURS = 12
RANDOM_SEED = 42

def discover_universe():
    base = 'https://www.nasdaqtrader.com/dynamic/SymDir/'
    frames = []
    for filename, symbol_col in [('nasdaqlisted.txt','Symbol'),('otherlisted.txt','ACT Symbol')]:
        try:
            response = requests.get(base+filename, timeout=20, headers={'User-Agent':'Mozilla/5.0'})
            response.raise_for_status()
            frame = pd.read_csv(io.StringIO(response.text), sep='|')
            frame = frame[(frame.get('Test Issue','N') == 'N') & (frame.get('ETF','N') == 'N')]
            frame = frame[frame[symbol_col].astype(str).str.fullmatch(r'[A-Z]{1,5}')]
            if 'Security Name' in frame:
                frame = frame[~frame['Security Name'].astype(str).str.contains('warrant|preferred|depositary|unit|rights',case=False,regex=True)]
            frames.extend(frame[symbol_col].astype(str).tolist())
        except Exception as e: print(f'Could not load {filename}: {e}')
    symbols = sorted(set(frames))
    if not symbols:
        print('Directory unavailable; using S&P 500 fallback (limited coverage).')
        try: symbols = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]['Symbol'].str.replace('.','-',regex=False).tolist()
        except Exception as e: raise RuntimeError('Could not discover symbols; check internet connection.') from e
    if SCAN_LIMIT and len(symbols) > SCAN_LIMIT:
        # Rotate subset each day so repeated runs gradually cover more companies.
        rng = np.random.default_rng(RANDOM_SEED + int(pd.Timestamp.now(tz='UTC').strftime('%Y%m%d')))
        symbols = sorted(rng.choice(symbols, size=SCAN_LIMIT, replace=False).tolist())
    print(f'Discovered {len(symbols):,} symbols for this run.')
    return symbols

UNIVERSE = discover_universe()

Discovered 1,500 symbols for this run.


# **Download and score stocks**

Uses daily closing prices (not intraday live ticks). Daily % change compares the two latest available daily closes. The score is a transparent 0–100 heuristic and does not forecast returns.

In [17]:

from pathlib import Path
import hashlib
from datetime import datetime, timezone, timedelta

# Cache individual batches, so re-running the notebook does not re-request
# recently downloaded data. These files are stored in the notebook's runtime.
SCAN_CACHE_DIR = Path('.stock_screener_scan_cache')
SCAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _scan_cache_path(chunk):
    key = hashlib.sha256(','.join(chunk).encode('utf-8')).hexdigest()[:20]
    return SCAN_CACHE_DIR / f'market_{key}.csv'

def _download_scan_chunk(chunk):
    data = yf.download(chunk, period='4mo', interval='1d', group_by='ticker',
                       auto_adjust=True, threads=True, progress=False)
    rows=[]; failed=[]
    for sym in chunk:
        try:
            df=data[sym].copy() if isinstance(data.columns,pd.MultiIndex) else data.copy()
            df=df.dropna(subset=['Close'])
            if len(df)<25: failed.append(sym); continue
            close=df['Close'].astype(float); volume=df['Volume'].astype(float)
            last=float(close.iloc[-1]); prev=float(close.iloc[-2])
            if last<=0 or prev<=0: failed.append(sym); continue
            avgvol=float(volume.iloc[-21:-1].mean())
            rv=float(volume.iloc[-1]/avgvol) if avgvol>0 else np.nan
            sma20=float(close.tail(20).mean())
            sma50=float(close.tail(50).mean()) if len(close)>=50 else np.nan
            ret20=float((last/close.iloc[-21]-1)*100)
            daily=close.pct_change().dropna()
            vol=float(daily.tail(60).std()*np.sqrt(252)*100)
            high20=float(df['High'].tail(20).max())
            low20=float(df['Low'].tail(20).min())
            rows.append(dict(Ticker=sym,Price=last,ChangePct=(last/prev-1)*100,
                             Volume=float(volume.iloc[-1]),AvgVolume20=avgvol,
                             RelVolume=rv,DollarVolume=last*float(volume.iloc[-1]),
                             SMA20=sma20,SMA50=sma50,Return20D=ret20,
                             VolatilityPct=vol,High20=high20,Low20=low20,
                             AsOf=str(df.index[-1].date())))
        except Exception:
            failed.append(sym)
    return pd.DataFrame(rows),failed

def download_market(symbols, chunk_size=None):
    chunk_size = chunk_size or SCAN_BATCH_SIZE
    chunks=[symbols[i:i+chunk_size] for i in range(0,len(symbols),chunk_size)]
    frames=[]; failed=[]; cached_count=0
    for index,chunk in enumerate(chunks,1):
        cache_file=_scan_cache_path(chunk)
        batch=None
        if SCAN_USE_CACHE and cache_file.exists():
            age_hours=(datetime.now(timezone.utc).timestamp()-cache_file.stat().st_mtime)/3600
            if 0<=age_hours<=SCAN_CACHE_HOURS:
                try:
                    batch=pd.read_csv(cache_file)
                    if 'Ticker' not in batch or 'Price' not in batch:
                        batch=None
                    else: cached_count+=1
                except Exception:
                    batch=None
        if batch is None:
            last_error=None
            for attempt in range(SCAN_MAX_RETRIES+1):
                try:
                    batch,missing=_download_scan_chunk(chunk)
                    if not batch.empty:
                        if SCAN_USE_CACHE: batch.to_csv(cache_file,index=False)
                        break
                    last_error='No usable data returned'
                except Exception as exc:
                    last_error=str(exc)[:130]
                if attempt<SCAN_MAX_RETRIES:
                    wait=min(20,SCAN_PAUSE_SECONDS*(2**(attempt+1)))
                    print(f'Batch {index} retry {attempt+1}: {last_error}; waiting {wait:.1f}s')
                    time.sleep(wait)
            if batch is None or batch.empty:
                failed.extend(chunk)
                print(f'Batch {index}/{len(chunks)} skipped after retries: {last_error}')
                continue
            time.sleep(SCAN_PAUSE_SECONDS)
        frames.append(batch)
        print(f'Batches {index}/{len(chunks)} | usable {sum(len(f) for f in frames):,} '
              f'| cached batches {cached_count}',end='\r')
    if not frames: raise RuntimeError('No quotes returned. Try a smaller scan or wait for provider access.')
    result=pd.concat(frames,ignore_index=True).drop_duplicates('Ticker',keep='last')
    print(f'\nUsable stocks: {len(result):,} / {len(symbols):,}; '
          f'cached batches: {cached_count}; unavailable: {len(symbols)-len(result):,}')
    return result

def add_scores(df):
    x=df.copy()
    # Rank within scanned universe: 35% 20-day momentum, 25% trend,
    # 20% relative volume, 20% liquidity. High volatility is a risk, not a bonus.
    trend=((x.Price>x.SMA20).astype(float)+(x.SMA20>x.SMA50).fillna(False).astype(float))/2
    momentum=x.Return20D.rank(pct=True).fillna(.5)
    relative=x.RelVolume.clip(0,5).rank(pct=True).fillna(.5)
    liquidity=x.AvgVolume20.rank(pct=True).fillna(.5)
    x['Score']=(100*(.35*momentum+.25*trend+.20*relative+.20*liquidity)).round(1)
    return x.sort_values('Score',ascending=False).reset_index(drop=True)

MARKET=add_scores(download_market(UNIVERSE))
print('Latest daily data:',MARKET.AsOf.max())
display(MARKET.head(15).round(2))

Batches 38/38 | usable 1,482 | cached batches 38
Usable stocks: 1,482 / 1,500; cached batches: 38; unavailable: 18
Latest daily data: 2026-10-22


,Ticker,Price,ChangePct,Volume,AvgVolume20,RelVolume,DollarVolume,SMA20,SMA50,Return20D,VolatilityPct,High20,Low20,AsOf,Score
0,NCPL,2.23,42.04,24063400.0,16641970.00,1.45,5.366138e+07,1.52,0.95,75.59,378.12,2.66,0.93,2026-10-22,95.6
1,KDP,35.19,2.09,28532300.0,11809300.00,2.42,1.004052e+09,32.36,31.82,14.78,27.34,35.40,30.06,2026-10-22,95.5
2,BRZE,45.86,0.39,9166200.0,5970195.00,1.54,4.203619e+08,29.94,29.26,85.89,129.10,45.94,22.27,2026-10-22,95.1
3,ACI,13.87,3.82,16997000.0,8351770.00,2.04,2.357484e+08,12.38,12.33,16.07,33.37,14.02,11.39,2026-10-22,95.0
4,NOW,157.00,2.53,17390000.0,10710705.00,1.62,2.730230e+09,140.51,136.85,13.95,50.97,157.60,126.75,2026-10-22,93.4
5,MYGN,5.76,3.41,4438000.0,2952195.00,1.50,2.556288e+07,4.45,3.83,39.13,133.41,5.95,3.95,2026-10-22,93.0
6,KMI,33.72,-0.74,31388600.0,12048540.00,2.61,1.058424e+09,32.26,31.86,7.90,23.16,34.60,29.43,2026-10-22,92.5
7,SITC,3.74,0.81,3890777.0,2296982.55,1.69,1.455151e+07,3.28,3.09,23.84,46.88,3.84,2.96,2026-10-22,92.4
8,SAP,240.59,3.38,5186400.0,2262810.00,2.29,1.247796e+09,217.49,214.89,15.05,28.70,243.05,203.17,2026-10-22,92.0
9,WMT,121.12,2.82,28309400.0,21200030.00,1.34,3.428835e+09,111.60,109.31,12.58,28.69,121.38,103.39,2026-10-22,91.6


# **Interactive screener, leaderboards and ticker deep dive**

Use the controls to select a price bucket, sort order, minimum liquidity, and ticker. Click Analyze ticker to load a full interactive chart. Changing filters refreshes the screener without downloading data again; Refresh market scan reruns the network scan.

In [18]:
def technicals(df):
    x=df.copy(); c=x['Close']; h=x['High']; l=x['Low']
    x['SMA20']=c.rolling(20).mean(); x['SMA50']=c.rolling(50).mean(); x['SMA200']=c.rolling(200).mean()
    x['EMA12']=c.ewm(span=12,adjust=False).mean(); x['EMA26']=c.ewm(span=26,adjust=False).mean()
    x['MACD']=x.EMA12-x.EMA26; x['Signal']=x.MACD.ewm(span=9,adjust=False).mean()
    delta=c.diff(); gain=delta.clip(lower=0).ewm(alpha=1/14,adjust=False,min_periods=14).mean()
    loss=(-delta.clip(upper=0)).ewm(alpha=1/14,adjust=False,min_periods=14).mean()
    rs=gain/loss.replace(0,np.nan); x['RSI']=100-100/(1+rs)
    x.loc[(loss==0)&(gain>0),'RSI']=100; x.loc[(loss==0)&(gain==0),'RSI']=50
    std=c.rolling(20).std(); x['UpperBB']=x.SMA20+2*std; x['LowerBB']=x.SMA20-2*std
    tr=pd.concat([h-l,(h-c.shift()).abs(),(l-c.shift()).abs()],axis=1).max(axis=1)
    x['ATR14']=tr.ewm(alpha=1/14,adjust=False,min_periods=14).mean()
    return x

def analyze_stock(ticker,period='1y'):
    ticker=ticker.strip().upper()
    if not ticker or not __import__('re').fullmatch(r'[A-Z0-9.\-^]{1,20}',ticker):
        print('Enter a valid ticker.'); return
    try:
        obj=yf.Ticker(ticker)
        raw=obj.history(period=period,interval='1d',auto_adjust=False)
        if raw.empty or len(raw)<30: print('Insufficient price history for',ticker); return
        df=technicals(raw)
        # Version 2.2: three independent chart panels, with separate titles and
        # legends. The horizontally scrollable HBox keeps all three columns.
        chart_style = dict(
            template='plotly_dark', paper_bgcolor='#111827',
            plot_bgcolor='#111827', height=490,
            font=dict(size=11), hovermode='x unified',
            margin=dict(l=58, r=24, t=125, b=62),
            legend=dict(orientation='h', x=0, y=1.08,
                        xanchor='left', yanchor='bottom',
                        font=dict(size=10)),
        )
        price_chart = go.Figure()
        price_chart.add_trace(go.Candlestick(
            x=df.index, open=df.Open, high=df.High, low=df.Low,
            close=df.Close, name='OHLC'))
        for col, color in [
            ('SMA20','#f59e0b'),('SMA50','#3b82f6'),
            ('SMA200','#a855f7'),('UpperBB','#9ca3af'),
            ('LowerBB','#9ca3af')
        ]:
            price_chart.add_trace(go.Scatter(
                x=df.index, y=df[col], name=col,
                line=dict(color=color, width=1.5,
                          dash='dot' if 'BB' in col else 'solid')))
        price_chart.update_layout(**chart_style,
                                  title=dict(text=f'{ticker} | Price & Trend',
                                             x=.02, y=.97, font=dict(size=16)))
        price_chart.update_xaxes(rangeslider_visible=False, title_text='Date')
        price_chart.update_yaxes(title_text='Price (USD)', automargin=True)

        volume_chart = go.Figure()
        volume_chart.add_trace(go.Bar(
            x=df.index, y=df.Volume, name='Daily volume',
            marker_color='#64748b'))
        volume_chart.add_trace(go.Scatter(
            x=df.index, y=df.Volume.rolling(20).mean(),
            name='20-day average', line=dict(color='#f59e0b', width=2)))
        volume_chart.update_layout(
            **chart_style,
            title=dict(text=f'{ticker} | Trading Volume',
                       x=.02, y=.97, font=dict(size=16)))
        volume_chart.update_xaxes(title_text='Date')
        volume_chart.update_yaxes(title_text='Shares', automargin=True)

        indicator_chart = make_subplots(
            rows=2, cols=1, shared_xaxes=True, vertical_spacing=.16,
            row_heights=[.48,.52])
        indicator_chart.add_trace(go.Scatter(
            x=df.index, y=df.RSI, name='RSI (14)',
            line=dict(color='#06b6d4')), row=1, col=1)
        for level in [30,70]:
            indicator_chart.add_hline(
                y=level, row=1, col=1, line_dash='dash',
                line_color='#64748b')
        indicator_chart.add_trace(go.Scatter(
            x=df.index, y=df.MACD, name='MACD',
            line=dict(color='#22c55e')), row=2, col=1)
        indicator_chart.add_trace(go.Scatter(
            x=df.index, y=df.Signal, name='Signal',
            line=dict(color='#f97316')), row=2, col=1)
        indicator_chart.add_trace(go.Bar(
            x=df.index, y=df.MACD-df.Signal, name='Histogram',
            marker_color='#94a3b8'), row=2, col=1)
        indicator_style={**chart_style,
                         'height':520,
                         'margin':dict(l=58,r=24,t=140,b=62)}
        indicator_chart.update_layout(
            **indicator_style,
            title=dict(text=f'{ticker} | RSI & MACD',
                       x=.02,y=.97,font=dict(size=16)))
        indicator_chart.update_yaxes(title_text='RSI', row=1, col=1,
                                     automargin=True)
        indicator_chart.update_yaxes(title_text='MACD', row=2, col=1,
                                     automargin=True)
        indicator_chart.update_xaxes(title_text='Date', row=2, col=1)

        # Full-width charts, one below another (not in columns).
        display(widgets.HTML(
            '<h3 style="margin:22px 0 12px">Technical charts</h3>'
            '<p style="margin:0 0 14px">Full-width price, volume and momentum charts.</p>'))
        for chart in (price_chart, volume_chart, indicator_chart):
            chart.update_layout(autosize=True, width=None)
            display(chart)
        try: info=obj.info or {}
        except Exception: info={}
        fields={'Company':'longName','Sector':'sector','Industry':'industry','Market cap':'marketCap',
                'Trailing P/E':'trailingPE','Forward P/E':'forwardPE','EPS':'trailingEps',
                'Dividend yield':'dividendYield','52-week high':'fiftyTwoWeekHigh',
                '52-week low':'fiftyTwoWeekLow','Beta':'beta','Profit margin':'profitMargins',
                'Revenue growth':'revenueGrowth','Debt/equity':'debtToEquity','Website':'website'}
        display(pd.DataFrame([(label,info.get(key,'N/A')) for label,key in fields.items()],columns=['Fundamental','Value']).set_index('Fundamental'))
        daily=df.Close.pct_change().dropna(); dd=(df.Close/df.Close.cummax()-1).min()*100
        atr=float(df.ATR14.iloc[-1]); last=float(df.Close.iloc[-1])
        risk=pd.DataFrame({'Metric':['Last close','RSI (14)','ATR (14)','ATR %','Annualized historical volatility','Max drawdown (selected period)','Illustrative 2×ATR below close'],
                           'Value':[last,float(df.RSI.iloc[-1]),atr,100*atr/last,100*daily.std()*np.sqrt(252),100*dd,last-2*atr]})
        display(risk.round(2).set_index('Metric'))
        print('ATR reference is educational, not a recommended stop-loss. Fundamentals may be missing or stale.')
    except Exception as e: print('Could not analyze ticker:',str(e)[:250])

In [20]:
# Unified dashboard: the clickable ticker buttons below open a detailed analysis IN the notebook.
# Fundamentals are retrieved ON DEMAND for a capped number of tickers to limit API requests.
import html
from datetime import datetime, timezone

FUNDAMENTAL_KEYS = {
    'Company':'longName', 'Sector':'sector', 'Industry':'industry',
    'MarketCap':'marketCap', 'EnterpriseValue':'enterpriseValue',
    'TrailingPE':'trailingPE', 'ForwardPE':'forwardPE', 'PriceToBook':'priceToBook',
    'PriceToSales':'priceToSalesTrailing12Months', 'EVtoEBITDA':'enterpriseToEbitda',
    'RevenueGrowth':'revenueGrowth', 'EarningsGrowth':'earningsGrowth',
    'ProfitMargin':'profitMargins', 'OperatingMargin':'operatingMargins',
    'ROE':'returnOnEquity', 'ROA':'returnOnAssets',
    'DebtToEquity':'debtToEquity', 'CurrentRatio':'currentRatio',
    'FreeCashFlow':'freeCashflow', 'OperatingCashFlow':'operatingCashflow',
    'DividendYield':'dividendYield', 'Beta':'beta'
}
FUND_CACHE = {}
FUND_ERRORS = {}

def get_fundamental(ticker, force=False):
    if ticker in FUND_CACHE and not force:
        return FUND_CACHE[ticker]
    try:
        info = yf.Ticker(ticker).info or {}
        record = {'Ticker':ticker, **{col:info.get(key, np.nan) for col,key in FUNDAMENTAL_KEYS.items()}}
        record['FundamentalsAsOfUTC'] = datetime.now(timezone.utc).isoformat(timespec='seconds')
        FUND_CACHE[ticker] = record
        FUND_ERRORS.pop(ticker,None)
        return record
    except Exception as exc:
        FUND_ERRORS[ticker] = str(exc)[:120]
        return None


# Version 2.4 scoring: all components are 0-100.
# Missing fundamentals are NOT treated as zero. Composite is N/A until sufficient data exists.
def bounded(value, low, high, invert=False):
    try:
        v=float(value)
        if not np.isfinite(v): return np.nan
        z=float(np.clip(100*(v-low)/(high-low),0,100))
        return 100-z if invert else z
    except (ValueError,TypeError): return np.nan

def quality_breakdown(record):
    """Five equally weighted quality metrics; at least 3 required."""
    if not record: return np.nan, 0, {}
    d={
        'Profit margin': bounded(record.get('ProfitMargin'),-.10,.25),
        'Revenue growth': bounded(record.get('RevenueGrowth'),-.20,.30),
        'Return on equity': bounded(record.get('ROE'),-.10,.30),
        'Debt-to-equity': bounded(record.get('DebtToEquity'),0,200,invert=True),
    }
    fcf=record.get('FreeCashFlow')
    try:
        fcf=float(fcf)
        d['Positive free cash flow']=100. if fcf>0 else 0. if np.isfinite(fcf) else np.nan
    except (TypeError,ValueError): d['Positive free cash flow']=np.nan
    valid=[v for v in d.values() if pd.notna(v)]
    return (round(float(np.mean(valid)),1) if len(valid)>=3 else np.nan,
            len(valid),d)

def advanced_scores(frame):
    x=frame.copy()
    x['TechnicalScore']=pd.to_numeric(x['Score'],errors='coerce')
    # Recompute the two original volume/liquidity percentiles as a standalone factor.
    x['VolumeLiquidityScore']=(
        .5*x['RelVolume'].clip(0,5).rank(pct=True)*100+
        .5*x['AvgVolume20'].rank(pct=True)*100).round(1)
    # Lower observed volatility and smaller recent high/low range score higher.
    volatility=pd.to_numeric(x['VolatilityPct'],errors='coerce')
    range_pct=(x['High20']-x['Low20'])/x['Price'].replace(0,np.nan)*100
    x['RiskScore']=(
        .7*(100-volatility.rank(pct=True)*100)+
        .3*(100-range_pct.rank(pct=True)*100)).round(1)
    quality=[quality_breakdown(FUND_CACHE.get(t)) for t in x['Ticker']]
    x['InvtQltyScore']=[item[0] for item in quality]
    x['QualityCoverage']=[f'{item[1]}/5' for item in quality]
    x['AdvStkScore']=(
        .35*x['TechnicalScore']+
        .30*x['InvtQltyScore']+
        .15*x['VolumeLiquidityScore']+
        .20*x['RiskScore']).round(1)
    return x

def finite_numeric(frame, col):
    return pd.to_numeric(frame[col], errors='coerce')

price_bucket = widgets.Dropdown(options=[('All prices',float('inf')),('Under $5',5),('Under $10',10),('Under $20',20),('Under $100',100)],value=float('inf'),description='Price')
rank_by = widgets.Dropdown(options=[('Opportunity score','Score'),('Advanced stock score','AdvStkScore'),('Investment quality score','InvtQltyScore'),('Top gainers','ChangePct'),('Top losers','Losers'),('Unusual volume','RelVolume'),('20-day momentum','Return20D')],description='Rank')
min_avg_volume = widgets.IntText(value=100000,description='Min 20d vol')
min_dollar_volume = widgets.FloatText(value=0,description='Min $ vol')
rows_to_show = widgets.IntSlider(value=15,min=5,max=30,step=5,description='Rows')
rv_threshold = widgets.FloatSlider(value=2.0,min=1.2,max=10,step=.2,description='RVOL alert',readout_format='.1f')
min_alert_volume = widgets.IntText(value=100000,description='Alert vol')
market_cap_min = widgets.FloatText(value=0,description='Min cap $M')
market_cap_max = widgets.FloatText(value=0,description='Max cap $M')
max_pe = widgets.FloatText(value=0,description='Max P/E')
min_revenue_growth = widgets.FloatText(value=-100,description='Rev growth %')
min_profit_margin = widgets.FloatText(value=-100,description='Profit margin %')
max_debt_equity = widgets.FloatText(value=0,description='Max D/E')
min_roe = widgets.FloatText(value=-100,description='Min ROE %')
sector_filter = widgets.Text(value='',description='Sector',placeholder='e.g. Technology')
require_fundamentals = widgets.Checkbox(value=False,description='Require fundamentals for filtering')
fund_batch_limit = widgets.IntSlider(value=20,min=5,max=60,step=5,description='Fetch max')
ticker_input = widgets.Text(value='AAPL',description='Ticker')
chart_period = widgets.Dropdown(options=['3mo','6mo','1y','2y','5y'],value='1y',description='History')
update_button = widgets.Button(description='Update dashboard',button_style='info')
refresh_button = widgets.Button(description='Refresh market prices',button_style='warning')
fund_button = widgets.Button(description='Load fundamentals for top matches',button_style='primary')
analyze_button = widgets.Button(description='Analyze ticker',button_style='success')
export_button = widgets.Button(description='Export filtered CSV')
status = widgets.Output(); overview = widgets.Output(); alerts_out = widgets.Output(); leaders = widgets.Output(); chart_out = widgets.Output()
clickable_area = widgets.VBox()
CURRENT = MARKET.copy()


def normalize_market(frame):
    x = frame.copy()
    if x.empty:
        raise ValueError("No stocks loaded. Rerun the market download cell.")
    if 'Ticker' not in x.columns and x.index.name == 'Ticker':
        x = x.reset_index()
    for col in ['Price', 'Volume']:
        if col not in x.columns:
            raise ValueError(f"Missing {col}. Rerun the market download cell.")
        x[col] = pd.to_numeric(x[col], errors='coerce')
    if 'DollarVolume' not in x.columns:
        x['DollarVolume'] = x['Price'] * x['Volume']
    if 'AvgVolume20' not in x.columns:
        x['AvgVolume20'] = np.nan
    for col in ['DollarVolume','AvgVolume20','RelVolume','ChangePct','Return20D','Score']:
        if col not in x.columns:
            x[col] = np.nan
        x[col] = pd.to_numeric(x[col], errors='coerce')
    if 'AsOf' not in x.columns:
        x['AsOf'] = 'unknown'
    return x

MARKET = normalize_market(MARKET)

# A threshold set to zero means the filter is disabled. Percent inputs use percent units.
def filter_market():
    global MARKET
    MARKET = normalize_market(MARKET)
    x=advanced_scores(MARKET)
    x=x[(MARKET.Price < price_bucket.value) & (MARKET.AvgVolume20 >= min_avg_volume.value) &
             (MARKET.DollarVolume >= min_dollar_volume.value)].copy()
    if require_fundamentals.value or any([market_cap_min.value,market_cap_max.value,max_pe.value,
       min_revenue_growth.value != -100,min_profit_margin.value != -100,max_debt_equity.value,
       min_roe.value != -100, bool(sector_filter.value.strip())]):
        # Only cached fundamentals can be screened. Do not silently assume missing fields pass.
        if not FUND_CACHE:
            return x.iloc[0:0].copy()
        f=pd.DataFrame(FUND_CACHE.values())
        x=x.merge(f,on='Ticker',how='inner')
        def apply(col, op, val):
            nonlocal x
            if col in x:
                x=x[op(finite_numeric(x,col),val).fillna(False)]
        if market_cap_min.value>0: apply('MarketCap',lambda a,b:a>=b,market_cap_min.value*1e6)
        if market_cap_max.value>0: apply('MarketCap',lambda a,b:a<=b,market_cap_max.value*1e6)
        if max_pe.value>0:
            apply('TrailingPE',lambda a,b:(a>0)&(a<=b),max_pe.value)
        if min_revenue_growth.value!=-100: apply('RevenueGrowth',lambda a,b:a>=b,min_revenue_growth.value/100)
        if min_profit_margin.value!=-100: apply('ProfitMargin',lambda a,b:a>=b,min_profit_margin.value/100)
        if max_debt_equity.value>0: apply('DebtToEquity',lambda a,b:a<=b,max_debt_equity.value)
        if min_roe.value!=-100: apply('ROE',lambda a,b:a>=b,min_roe.value/100)
        if sector_filter.value.strip():
            x=x[x.Sector.fillna('').str.contains(sector_filter.value.strip(),case=False,regex=False)]
    key=rank_by.value
    return x.sort_values('ChangePct',ascending=True) if key=='Losers' else x.sort_values(key,ascending=False,na_position='last')

def open_ticker(ticker):
    ticker_input.value=ticker
    with chart_out:
        clear_output(wait=True)
        print('Loading detailed research for',ticker)
        analyze_stock(ticker,chart_period.value)
        cached=get_fundamental(ticker)
        if cached:
            company=cached.get('Company')
            sector=cached.get('Sector')
            industry=cached.get('Industry')
            company=company if isinstance(company,str) and company.strip() else 'Not available'
            sector=sector if isinstance(sector,str) and sector.strip() else 'Not available'
            industry=industry if isinstance(industry,str) and industry.strip() else 'Not available'
            display(widgets.HTML(
                '<div style="padding:14px 18px;border:1px solid #64748b;'
                'border-radius:10px;margin:12px 0">'
                '<h3 style="margin:0 0 8px">'+html.escape(company)+' ('+html.escape(ticker)+')</h3>'
                '<b>Sector:</b> '+html.escape(sector)+
                ' &nbsp; | &nbsp; <b>Industry:</b> '+html.escape(industry)+'</div>'))
            q,count,parts=quality_breakdown(cached)
            row=advanced_scores(MARKET)
            row=row[row.Ticker==ticker]
            display(widgets.HTML('<h3>Investment quality and combined scoring</h3>'))
            if row.empty:
                print('Investment Quality:',q if pd.notna(q) else 'N/A',
                      '|',count,'of 5 quality metrics available')
                print('Combined score requires the ticker in the scanned market universe.')
            else:
                r=row.iloc[0]
                for name,col in [('Technical','TechnicalScore'),
                                 ('Investment Quality','InvtQltyScore'),
                                 ('Volume & Liquidity','VolumeLiquidityScore'),
                                 ('Risk Profile','RiskScore'),
                                 ('Advanced Stock Score','AdvStkScore')]:
                    val=r[col]
                    print(f'{name}: {val:.1f}/100' if pd.notna(val) else f'{name}: N/A')
                print('Quality metrics available:',r.QualityCoverage)
            display(pd.DataFrame({'Quality metric':list(parts),
                                  'Component score':list(parts.values())}).round(1))
            display(pd.DataFrame({'Metric':list(cached.keys()),
                                  'Value':list(cached.values())}).set_index('Metric'))
        else:
            print('Company name, sector, and fundamentals unavailable from data provider.')

def ticker_buttons(frame, limit=20):
    buttons=[]
    for sym in frame.Ticker.head(limit):
        b=widgets.Button(description=str(sym),layout=widgets.Layout(width='92px'),tooltip='Analyze '+str(sym))
        b.on_click(lambda _,s=sym:open_ticker(s))
        buttons.append(b)
    return widgets.GridBox(buttons,layout=widgets.Layout(grid_template_columns='repeat(auto-fill, 100px)',grid_gap='5px'))

def render_dashboard(_=None):
    global CURRENT
    CURRENT=filter_market()
    scored_market=advanced_scores(MARKET)
    columns=['Ticker','Price','ChangePct','Volume','AvgVolume20','RelVolume','DollarVolume','Return20D','Score','InvtQltyScore','AdvStkScore']
    extras=[c for c in ['Sector','MarketCap','TrailingPE','RevenueGrowth','ProfitMargin','ROE','DebtToEquity','FreeCashFlow'] if c in CURRENT.columns]
    with overview:
        clear_output(wait=True)
        display(HTML('<h3>Screening results</h3>'))
        print(f'{len(CURRENT):,} matches / {len(MARKET):,} scanned | session: {MARKET.AsOf.max()} | fundamentals cached: {len(FUND_CACHE)}')
        print('Load fundamentals for top matches to populate Investment Quality and Advanced scores.')
        if CURRENT.empty: print('No matches. If using fundamental filters, load fundamentals first and note only cached symbols are covered.')
        else:
            display(CURRENT[columns+extras].head(rows_to_show.value).round(2))
            # Two additional market overview charts (based on screened data).
            movers=CURRENT.dropna(subset=['ChangePct']).copy()
            if not movers.empty:
                movers=movers.reindex(movers.ChangePct.abs().nlargest(12).index)
                movers=movers.sort_values('ChangePct')
                movers_chart=go.Figure(go.Bar(
                    x=movers.ChangePct, y=movers.Ticker, orientation='h',
                    marker_color=['#ef4444' if v<0 else '#22c55e'
                                  for v in movers.ChangePct]))
                movers_chart.update_layout(
                    title='Largest daily moves among current matches',
                    template='plotly_dark', height=390,
                    margin=dict(l=80,r=30,t=65,b=45),
                    xaxis_title='Daily change (%)', yaxis_title='')
                display(movers_chart)
            volume_data=CURRENT.dropna(subset=['RelVolume']).nlargest(12,'RelVolume')
            if not volume_data.empty:
                volume_chart=go.Figure(go.Bar(
                    x=volume_data.Ticker, y=volume_data.RelVolume,
                    marker_color='#38bdf8'))
                volume_chart.add_hline(y=2, line_dash='dash',
                                       line_color='#f59e0b',
                                       annotation_text='2× normal')
                volume_chart.update_layout(
                    title='Highest relative volume among current matches',
                    template='plotly_dark', height=350,
                    margin=dict(l=60,r=25,t=65,b=55),
                    yaxis_title='Relative volume (×)')
                display(volume_chart)
    clickable_area.children=(widgets.HTML('<b>Click a ticker to load its chart and company analysis:</b>'),ticker_buttons(CURRENT,rows_to_show.value))
    with alerts_out:
        clear_output(wait=True)
        display(HTML('<h3>Unusual-volume alerts (daily bars)</h3>'))
        a=scored_market[(scored_market.RelVolume>=rv_threshold.value)&(scored_market.Volume>=min_alert_volume.value)].sort_values('RelVolume',ascending=False)
        print(f'{len(a)} alert(s) | threshold {rv_threshold.value:.1f}x prior 20-session average volume')
        display(a[columns].head(25).round(2))
        print('Alert = detected on latest available daily bar; NOT a push notification. During market hours, partial-day volume can distort comparisons.')
    with leaders:
        clear_output(wait=True)
        display(HTML('<h3>Market leaders and price tiers</h3>'
                     '<p>Six tables in three side-by-side columns. '
                     'Scroll horizontally if your screen is narrow.</p>'))
        leader_specs = [
            ('Top gainers', scored_market.nlargest(10,'ChangePct')),
            ('Top losers', scored_market.nsmallest(10,'ChangePct')),
            ('Top scored under $5', scored_market[scored_market.Price<5].nlargest(10,'Score')),
            ('Top scored under $10', scored_market[scored_market.Price<10].nlargest(10,'Score')),
            ('Top scored under $20', scored_market[scored_market.Price<20].nlargest(10,'Score')),
            ('Top scored under $100', scored_market[scored_market.Price<100].nlargest(10,'Score')),
        ]
        leader_cards = []
        for title, frame in leader_specs:
            card = widgets.Output(layout=widgets.Layout(
                width='100%', min_width='0',
                border='1px solid #d6dce5', padding='8px'))
            with card:
                display(HTML('<h4 style="margin:4px 0 10px">'+title+'</h4>'))
                # Hide the arbitrary source row index to reduce crowding.
                display(HTML(frame[['Ticker','Price','ChangePct','RelVolume','Score','InvtQltyScore','AdvStkScore']]
                             .round(2).to_html(index=False, border=0)))
            leader_cards.append(card)
        leaders_grid = widgets.GridBox(
            leader_cards,
            layout=widgets.Layout(
                width='100%', min_width='1110px',
                grid_template_columns='repeat(3, minmax(350px, 1fr))',
                grid_gap='14px', align_items='flex-start'))
        display(widgets.Box(
            [leaders_grid], layout=widgets.Layout(
                width='100%', overflow_x='auto')))

def load_fundamentals(_):
    # Fetch the leading technical matches, not all US listings. This is an explicit partial coverage universe.
    # Compute the optional scores before sorting. These are not raw MARKET columns.
    # Loading by Advanced/Quality Score creates a bootstrap problem when no
    # fundamentals have been fetched yet; use Technical Score to seed coverage.
    base=advanced_scores(normalize_market(MARKET))
    base=base[(base['Price']<price_bucket.value) &
              (base['AvgVolume20']>=min_avg_volume.value)].copy()
    key=rank_by.value
    if key == 'Losers':
        base=base.sort_values('ChangePct',ascending=True,na_position='last')
    else:
        if key not in base.columns:
            key='Score'
        if key in ('AdvStkScore','InvtQltyScore'):
            # Previously fetched symbols can be ranked on quality, but unknown
            # companies need technical ranking to get a fair chance to load.
            base['_priority']=pd.to_numeric(base[key],errors='coerce')
            base['_seed']=pd.to_numeric(base['Score'],errors='coerce')
            base=base.sort_values(['_priority','_seed'],ascending=False,
                                  na_position='last')
            # If all quality data are missing, sort by technical score instead.
            if base['_priority'].notna().sum()==0:
                base=base.sort_values('_seed',ascending=False,na_position='last')
        else:
            base=base.sort_values(key,ascending=False,na_position='last')
    symbols=[t for t in base['Ticker'] if t not in FUND_CACHE][:fund_batch_limit.value]
    if not symbols:
        with status:
            clear_output(wait=True)
            print('All matching stocks have already been cached. Change the filters or refresh to load others.')
        render_dashboard()
        return
    with status:
        clear_output(wait=True)
        print(f'Fetching fundamentals for {len(symbols)} symbols. Data may be stale, missing, or rate-limited.')
        for i,sym in enumerate(symbols,1):
            get_fundamental(sym)
            if i%5==0: print(f'{i}/{len(symbols)} fetched or attempted')
            time.sleep(.15)
        print(f'Cached: {len(FUND_CACHE)} | failed: {len(FUND_ERRORS)}')
    render_dashboard()

def refresh_market(_):
    global MARKET,UNIVERSE
    with status:
        clear_output(wait=True)
        print('Refreshing quotes...')
        try:
            UNIVERSE=discover_universe()
            MARKET=normalize_market(add_scores(download_market(UNIVERSE)))
            print('Updated',MARKET.AsOf.max())
        except Exception as exc: print('Refresh failed; old data retained:',str(exc)[:180])
    render_dashboard()

def export_csv(_):
    filename='ultimate_screener_filtered.csv'
    CURRENT.to_csv(filename,index=False)
    with status:
        print('Saved:',filename)
        try:
            from google.colab import files
            files.download(filename)
        except ImportError: pass

update_button.on_click(render_dashboard)
refresh_button.on_click(refresh_market)
fund_button.on_click(load_fundamentals)
analyze_button.on_click(lambda _:open_ticker(ticker_input.value.strip().upper()))
export_button.on_click(export_csv)

display(widgets.HTML('<h1>📈 Ultimate Market Intelligence Terminal</h1><p>One dashboard: discovery, alerts, financial filters, clickable ticker research.</p>'))
display(widgets.HTML('<h3>Market screening</h3>'))
display(widgets.HBox([price_bucket,rank_by,min_avg_volume]),widgets.HBox([min_dollar_volume,rows_to_show]))
display(widgets.HTML('<h3>Advanced fundamental filters (on-demand cached coverage)</h3><p><b>Advanced score:</b> Technical 35% + Investment Quality 30% + Volume/Liquidity 15% + Risk 20%. Investment Quality requires at least 3 of 5 metrics. Missing data produces N/A, not a misleading zero.</p>'))
display(widgets.HBox([market_cap_min,market_cap_max,max_pe]),widgets.HBox([min_revenue_growth,min_profit_margin,max_debt_equity]),widgets.HBox([min_roe,sector_filter,require_fundamentals]))
display(widgets.HBox([fund_batch_limit,fund_button]))
display(widgets.HBox([update_button,refresh_button,export_button]),status)
display(overview,clickable_area)
display(widgets.HTML('<h3>Volume alert settings</h3>'))
display(widgets.HBox([rv_threshold,min_alert_volume]),alerts_out)
display(leaders)
display(widgets.HTML('<h3>Individual ticker research</h3>'))
display(widgets.HBox([ticker_input,chart_period,analyze_button]),chart_out)
render_dashboard()


HTML(value='<h1>📈 Ultimate Market Intelligence Terminal</h1><p>One dashboard: discovery, alerts, financial fil…

HTML(value='<h3>Market screening</h3>')

HTML(value='<h3>Advanced fundamental filters (on-demand cached coverage)</h3><p><b>Advanced score:</b> Technic…

Output()

Output()

...

HTML(value='<h3>Volume alert settings</h3>')

Output()

Output()

HTML(value='<h3>Individual ticker research</h3>')

Output()